In [ ]:
import sys
import psycopg2

sys.path.append('..')
from config import get_db_connection_params

print("🧹 Starting Data Cleaning Process...")

try:
    params = get_db_connection_params()
    conn = psycopg2.connect(**params)
    cursor = conn.cursor()
    print("✅ Connected to the database successfully.")
except Exception as e:
    print("❌ Connection error:", e)

🧹 Starting Data Cleaning Process...
✅ Connected to the database successfully.


In [ ]:
try:
    with open('../sql/data_cleaning.sql', 'r', encoding='utf-8') as file:
        cleaning_queries = file.read()

    cursor.execute(cleaning_queries)
    conn.commit()
    
    print("✅ Data cleaning executed successfully!")
    print("   - Whitespaces removed and text standardized (LOWER/TRIM).")
    print("   - Exact duplicate records removed using CTID.")
except Exception as e:
    print("❌ Error during data cleaning:", e)
    conn.rollback()

✅ Data cleaning executed successfully!
   - Whitespaces removed and text standardized (LOWER/TRIM).
   - Exact duplicate records removed using CTID.


In [ ]:
if 'cursor' in locals() and cursor:
    cursor.close()
if 'conn' in locals() and conn:
    conn.close()
    
print("🔒 Database connection closed.")

🔒 Database connection closed.


In [ ]:
import sys
import psycopg2

sys.path.append('..')
from config import get_db_connection_params

print("⚙️ Building Feature Store Table...")

try:
    params = get_db_connection_params()
    conn = psycopg2.connect(**params)
    cursor = conn.cursor()

    with open('../sql/feature_store.sql', 'r', encoding='utf-8') as file:
        feature_store_sql = file.read()

    cursor.execute(feature_store_sql)
    conn.commit()
    print("✅ Feature store table 'feature_store_olist' created successfully!")

except Exception as e:
    print("❌ Error creating feature store:", e)
    conn.rollback()

⚙️ Building Feature Store Table...
✅ Feature store table 'feature_store_olist' created successfully!


In [5]:
# Check target balance and baseline record counts in feature store
cursor.execute("""
    SELECT 
        is_late_delivery,
        COUNT(*) AS total_orders,
        ROUND(COUNT(*)::DECIMAL / SUM(COUNT(*)) OVER () * 100, 2) AS percentage
    FROM feature_store_olist
    GROUP BY is_late_delivery;
""")

results = cursor.fetchall()

print("📊 Target Variable Distribution (is_late_delivery):")
print("-" * 55)
for row in results:
    status = "Late Delivery (1)" if row[0] == 1 else "On-time Delivery (0)"
    print(f"• {status:<25} : {row[1]:,} orders ({row[2]}%)")

📊 Target Variable Distribution (is_late_delivery):
-------------------------------------------------------
• On-time Delivery (0)      : 74,978 orders (91.89%)
• Late Delivery (1)         : 6,618 orders (8.11%)


In [ ]:
cursor.close()
conn.close()
print("🔒 Connection closed successfully.")

🔒 Connection closed successfully.
